# Math 10 · Neural Networks: Function Composition, Backpropagation, Initialization, and Gradient Flow

**Companion lessons:** Lessons 19-30, Papers 02, 09, 19-21

> **Math goal:** understand the objects, assumptions, derivation, and failure modes behind the algorithms, not just memorize formulas.

## Layer as affine map plus nonlinearity

For layer $l$:

$$
z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)}
$$

$$
a^{(l)}=\phi(z^{(l)}).
$$

A deep network is repeated function composition.

## Backpropagation

For a scalar loss $L$:

$$
\delta^{(l)}
=
\frac{\partial L}{\partial z^{(l)}}.
$$

Then

$$
\frac{\partial L}{\partial W^{(l)}}
=
\delta^{(l)}(a^{(l-1)})^T
$$

under column-vector convention.

And

$$
\delta^{(l-1)}
=
(W^{(l)})^T\delta^{(l)}
\odot
\phi'(z^{(l-1)}).
$$

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import numpy as np
rng=np.random.default_rng(0)
a0=rng.normal(size=(4,1))
W1=rng.normal(size=(3,4)); b1=np.zeros((3,1))
W2=rng.normal(size=(2,3)); b2=np.zeros((2,1))
z1=W1@a0+b1; a1=np.maximum(0,z1); z2=W2@a1+b2
delta2=np.ones_like(z2)
dW2=delta2@a1.T
delta1=(W2.T@delta2)*(z1>0)
dW1=delta1@a0.T
print(dW1.shape,dW2.shape)

## Variance propagation

If weights are too large, activations/gradients can explode. If too small, they can vanish.

He initialization for ReLU commonly uses

$$
\operatorname{Var}(W_{ij})\approx\frac{2}{n_{in}}.
$$

In [ ]:
import numpy as np
rng=np.random.default_rng(1)
for scale_name,scale in [("tiny",.01),("He",np.sqrt(2/100)),("huge",1.0)]:
    x=rng.normal(size=(10000,100))
    W=rng.normal(0,scale,size=(100,100))
    y=np.maximum(0,x@W)
    print(scale_name,"input var",x.var(),"output var",y.var())

## Residual Jacobian

For a residual block

$$
y=x+F(x),
$$

the Jacobian is

$$
\frac{\partial y}{\partial x}
=
I+J_F.
$$

The identity term creates a direct gradient path.

### Activity
Multiply random Jacobians for a deep plain chain and compare singular values with residual Jacobians `I + J`. Observe vanishing/exploding behavior.

## Derivation checkpoint
In a new Markdown cell, re-derive the main result without copying the notebook. State every variable's shape and every assumption used.

## Engineering checkpoint
Explain which approximation or assumption is most likely to break in a real system, and how you would detect that failure from data.